# Alice in Wonderland

Following code (written by Jason Brownlee) is a simple LSTM-based RNN that generates text after training on the text of Alice in Wonderland. Work your way through the code (you can use this linked guide as a reference) to understand how the model is set up and how it's predicting the next character.



Note: When you run this it may say that it's going to take hours to train. No biggie. Open a local Jupyter notebook. It should run much quicker!



In [1]:
# Larger LSTM Network to Generate Text for Alice in Wonderland
!pip install tensorflow
import numpy
import urllib.request
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import LSTM
from keras.callbacks import ModelCheckpoint
from tensorflow.python.keras.utils.np_utils import to_categorical

# load ascii text from gist and covert to lowercase
url = "https://gist.githubusercontent.com/phillipj/4944029/raw/75ba2243dd5ec2875f629bf5d79f6c1e4b5a8b46/alice_in_wonderland.txt"

with urllib.request.urlopen(url) as response:
	raw_text = response.read()

raw_text = raw_text.lower()
# create mapping of unique chars to integers
chars = sorted(list(set(raw_text)))
char_to_int = dict((c, i) for i, c in enumerate(chars))
# summarize the loaded data
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 16.1 MB/s  0:00:19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 28.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [tensorflow]


/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Total Characters:  148574
Total Vocab:  46


In [2]:
# prepare the dataset of input to output pairs encoded as integers
seq_length = 100
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
	seq_in = raw_text[i:i + seq_length]
	seq_out = raw_text[i + seq_length]
	dataX.append([char_to_int[char] for char in seq_in])
	dataY.append(char_to_int[seq_out])
n_patterns = len(dataX)
print ("Total Patterns: ", n_patterns)
# reshape X to be [samples, time steps, features]
X = numpy.reshape(dataX, (n_patterns, seq_length, 1))
# normalize
X = X / float(n_vocab)
# one hot encode the output variable
y = to_categorical(dataY)
# define the LSTM model
model = Sequential()
model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(256))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')

Total Patterns:  148474


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [3]:
#define the checkpoint. This will let you reload old weights.
#MOST IMPORTNT CELL
filepath="weights-improvement-{epoch:02d}-{loss:.4f}-bigger.hdf5.keras"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
callbacks_list = [checkpoint]
# fit the model


In [4]:
# fit the model

model.fit(X, y, epochs=10, batch_size=512, callbacks=callbacks_list)



Epoch 1/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 3.0967
Epoch 1: loss improved from None to 3.01180, saving model to weights-improvement-01-3.0118-bigger.hdf5.keras

Epoch 1: finished saving model to weights-improvement-01-3.0118-bigger.hdf5.keras
290/290 ━━━━━━━━━━━━━━━━━━━━ 722s 2s/step - loss: 3.0118
Epoch 2/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 2.7866
Epoch 2: loss improved from 3.01180 to 2.73074, saving model to weights-improvement-02-2.7307-bigger.hdf5.keras

Epoch 2: finished saving model to weights-improvement-02-2.7307-bigger.hdf5.keras
290/290 ━━━━━━━━━━━━━━━━━━━━ 717s 2s/step - loss: 2.7307
Epoch 3/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 2.6223
Epoch 3: loss improved from 2.73074 to 2.59001, saving model to weights-improvement-03-2.5900-bigger.hdf5.keras

Epoch 3: finished saving model to weights-improvement-03-2.5900-bigger.hdf5.keras
290/290 ━━━━━━━━━━━━━━━━━━━━ 714s 2s/step - loss: 2.5900
Epoch 4/10
290/290 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/ste

KeyboardInterrupt: 

## Predict step
Now that the model is trained, you can use the code below to predict sing the code below to predict the next letter in the sequence.


In [5]:
import sys
int_to_char = dict((i, c) for i, c in enumerate(chars))
# pick a random seed
start = numpy.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Seed:")
print("\"", ''.join([chr(int_to_char[value]) for value in pattern]), "\"")
# generate characters
for i in range(600):
	x = numpy.reshape(pattern, (1, len(pattern), 1))
	x = x / float(n_vocab)
	prediction = model.predict(x, verbose=0)
	index = numpy.argmax(prediction)
	result = int_to_char[index]
	seq_in = [int_to_char[value] for value in pattern]
	sys.stdout.write(chr(result))
	if i%100 == 0:
		sys.stdout.write("\n")
	pattern.append(index)
	pattern = pattern[1:len(pattern)]
print("\nDone.")

Seed:
" she had not as
yet had any dispute with the queen, but she knew that it might
happen any minute, `an "
d
 the was to the wout the woue to the woue to the woue to the woue to the cooee to the cooee to the h
ort the woue to the cooee to the cooee to the hort the woue to the cooee to the hort the woue to the
 cooee to the cooee to the hort the woue to the cooee to the hort the woue to the cooee to the cooee
 to the hort the woue to the cooee to the hort the woue to the cooee to the cooee to the hort the wo
ue to the cooee to the hort the woue to the cooee to the cooee to the hort the woue to the cooee to 
the hort the woue to the cooee to the cooee to the hort the woue to the cooee to the hort the woue 
Done.


In [6]:
#print(int_to_char)
#print(pattern)
print(chr(chars[10]))


.


In [7]:
# load the network weights
filename = "weights-improvement-02-2.3952-bigger.hdf5.keras"
model.load_weights(filename)
model.compile(loss='categorical_crossentropy', optimizer='adam')
# pick a random seed
start = np.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Seed:")
print("\"", ''.join(([chr(int_to_char[value]) for value in pattern])), "\"")
# generate characters
for i in range(1000):
	x = np.reshape(pattern, (1, len(pattern), 1))
	x = x / float(n_vocab)
	prediction = model.predict(x, verbose=0)
	index = np.argmax(prediction)
	result = int_to_char[index]
	seq_in = [int_to_char[value] for value in pattern]
	sys.stdout.write(chr(result))
	pattern.append(index)
	pattern = pattern[1:len(pattern)]
print("\nDone.")

FileNotFoundError: [Errno 2] No such file or directory: 'weights-improvement-02-2.3952-bigger.hdf5.keras'

In [8]:
print(int_to_char)

{0: 10, 1: 32, 2: 33, 3: 34, 4: 39, 5: 40, 6: 41, 7: 42, 8: 44, 9: 45, 10: 46, 11: 48, 12: 51, 13: 58, 14: 59, 15: 63, 16: 91, 17: 93, 18: 95, 19: 96, 20: 97, 21: 98, 22: 99, 23: 100, 24: 101, 25: 102, 26: 103, 27: 104, 28: 105, 29: 106, 30: 107, 31: 108, 32: 109, 33: 110, 34: 111, 35: 112, 36: 113, 37: 114, 38: 115, 39: 116, 40: 117, 41: 118, 42: 119, 43: 120, 44: 121, 45: 122}
